# 流程

```markdown
📦 主训练流程
├── 🛠️ 初始化阶段
│   ├── 📄 参数解析 (argparse)
│   │   ├── 训练参数: epochs, batch_size, learning_rate
│   │   ├── 设备配置: device, dtype, ddp
│   │   ├── 优化参数: grad_clip, accumulation_steps
│   │   └── 模型配置: dim, n_layers, max_seq_len, use_moe
│   │
│   ├── 🌐 分布式初始化
│   │   ├── NCCL 后端初始化 (init_distributed_mode)
│   │   ├── 设备绑定 (cuda:local_rank)
│   │   └️ DDP 包装模型
│   │
│   ├️ 🧠 模型初始化
│   │   ├️ 加载 tokenizer (AutoTokenizer)
│   │   └️ 创建 MiniMindLM 模型
│   │
│   └️ 📊 数据准备
│       ├️ 数据集加载 (PretrainDataset)
│       ├️ 分布式采样器 (DistributedSampler)
│       └️ 数据加载器 (DataLoader)
│
├️ 🔁 训练主循环 (train_epoch)
│   ├️ 🔄 数据迭代
│   │   ├️ 设备转移 (X, Y, loss_mask → cuda)
│   │   └️ 动态学习率 (cosine 衰减)
│   │
│   ├️ 🧮 计算过程
│   │   ├️ 混合精度上下文 (autocast)
│   │   ├️ 前向传播 (model(X))
│   │   ├️ 损失计算
│   │   │   ├️ 交叉熵损失 (CrossEntropyLoss)
│   │   │   └️ 辅助损失合并
│   │   │
│   │   └️ 反向传播
│   │       ├️ 梯度缩放 (GradScaler)
│   │       ├️ 梯度裁剪 (clip_grad_norm_)
│   │       └️ 参数更新 (AdamW)
│   │
│   └️ 📈 监控与保存
│       ├️ 训练日志 (Logger)
│       │   ├️ 损失值/lr/剩余时间
│       │   └️ WandB 集成
│       │
│       └️ 模型保存
│           ├️ DDP 模型状态提取
│           └️ 检查点保存 (间隔保存)
│
└️ ⚙️ 训练后处理
    ├️ 分布式进程清理
    └️ 资源释放
```



# 一些概念

## 损失函数

损失函数就像作文批改老师：
当模型（学生）预测出一个句子时，损失函数会逐字检查每个预测词是否正确，并给出扣分（损失值）。CrossEntropyLoss 就是常用的"评分标准"。

## 学习率

假设你在教一个婴儿走路（模型训练过程）


| 概念 | 类比 | 作用机制 |
|-------------|-------------------|--------------------------------------------------------------------------|
| 损失函数 | 跌倒检测器 | 当婴儿摔倒时（预测错误），会发出哭声（损失值）指示错误程度 |
| 学习率 | 步伐调整策略 | 决定每次根据哭声（损失）调整步伐的幅度： |
| | | - 🚶大步走：快速进步但容易摔倒（学习率过大→震荡） |
| | | - 🐌小碎步：稳定但进步慢（学习率过小→收敛慢） |
| | | - 🎯智能步：先大后小（余弦退火→快速接近目标+精细调整） |


```python
# 伪代码示例
for data in dataset:
    prediction = model(data)          # 婴儿迈出一步(预测)
    loss = loss_function(prediction)  # 检测是否摔倒(计算损失)
    
    gradients = compute_gradients(loss)  # 分析摔倒原因（计算梯度）
    
    # 学习率控制调整幅度 ↓↓↓
    parameters -= learning_rate * gradients  # 根据分析结果调整步伐
```
